In [93]:
import numpy as np
import math
import pandas as pd 
import scipy.stats as stats 
import matplotlib.pyplot as plt
from numba.typed import List
import itertools
from scipy.stats import linregress
from numba import njit
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
from itertools import product

In [94]:

def give_all_comb(len):
    return [seq  for i, seq in enumerate(product([-1, 1], repeat=len))]


In [95]:
@njit
def seq_to_index(context):
    idx = 0
    for i in range(len(context)):
        idx = idx * 2 + (context[i] == 1)
    return idx

@njit
def homm_get_next_w(context_arr, memorySeq,memoryStrengthPlus, memoryStrengthMinus, order, gamma):
    act_pos = 0.0
    act_neg = 0.0
    lag_weights = np.exp(-gamma * (order - 1 - np.arange(order)))

    for i in range(len(memorySeq)):
        memory   = memorySeq[i]
        abs_dif  = lag_weights * np.abs(memory - context_arr)
        sim      = np.exp(-np.sum(abs_dif))
        strengthPlus = memoryStrengthPlus[i]
        strengthMinus = memoryStrengthMinus[i]

        act_pos += strengthPlus * sim
        act_neg += strengthMinus * sim  # negative -1 evidence → positive +1 evidence

    total = act_pos + act_neg
    if total > 0.0:
        return act_pos / total
    return 0.5

@njit
def homm_add_to_memory(context_arr,resp, confidence, memoryStrengthPlus, memoryStrengthMinus, decay):
    idx = seq_to_index(context_arr)
    memoryStrengthPlus *= decay
    memoryStrengthMinus *= decay

    if resp ==1:
        memoryStrengthPlus[idx] += confidence
    else:
        memoryStrengthMinus[idx] += confidence

@njit
def build_context(resp_arr, i, k):
    context = np.zeros(k, dtype=np.int64)
    start = max(i - k, 0)
    length = i - start
    for j in range(length):
        context[k - length + j] = resp_arr[start + j]
    return context

def normalize_weights(weights, k_w):

    weights = np.asarray(weights, dtype=float)
    target_sum = 2 ** (k_w - 1)
    weights = target_sum * weights / np.sum(weights)

    for _ in range(100):
        excess = np.maximum(0, weights - 1)
        if np.sum(excess) < 1e-12:
            break                                   # all weights in [0, 1]
        weights = np.minimum(weights, 1)            # clip
        good = weights < 1
        if not np.any(good):
            break                                   # no room to redistribute
        weights[good] += np.sum(excess) / np.sum(good)

    return weights

In [96]:
import numpy as np
from numba import njit
from scipy.stats import norm as _norm

# Pre-computed 5-point Gauss-Legendre nodes and weights on [-1, 1]
_GL5_NODES   = np.array([-0.9061798459386640, -0.5384693101056831,
                           0.0,
                           0.5384693101056831,  0.9061798459386640])
_GL5_WEIGHTS = np.array([ 0.2369268850561891,  0.4786286704993665,
                           0.5688888888888889,
                           0.4786286704993665,  0.2369268850561891])

@njit
def _norm_pdf(x):
    return np.exp(-0.5 * x * x) / np.sqrt(2.0 * np.pi)

@njit
def _ddm_pdf_lower_core(t, v, a, w, sv, max_J=50, eps=1e-12):
    """
    Core lower-boundary PDF for a single (v, a, w) triple.
    σ is assumed = 1 here; caller must pre-scale v, a, sv by 1/σ.
    sv = inter-trial drift SD  (set 0 for no drift variability).
    w  = relative starting point in (0,1).
    """
    if t <= 0.0:
        return 1e-300
    if t < 1e-6:
        t = 1e-6

    one_plus = 1.0 + sv * sv * t
    denom = np.sqrt(t**3 * one_plus)

    expo = np.exp(
        (-(v * v) * t - 2.0 * v * a * w + (sv * sv) * (a * w) * (a * w))
        / (2.0 * one_plus)
    )

    sqrt_t = np.sqrt(t)
    s = 0.0
    prev_term = 0.0

    for j in range(max_J):
        if j % 2 == 0:
            rj = j * a + a * w
        else:
            rj = j * a + a * (1.0 - w)

        term = ((-1) ** j) * rj * _norm_pdf(rj / sqrt_t)
        s += term

        if j > 3 and np.abs(term) < eps and np.abs(prev_term) < eps:
            break
        prev_term = term

    pdf = (expo / denom) * s
    return max(pdf, 1e-300)


@njit
def ddm_pdf_lower_boundary(t, v, a, w, t0,
                            sv=0.0, sz=0.0, sigma=1.0,
                            max_J=50, eps=1e-12):
    """
    Lower-boundary PDF with:
      σ   (sigma)  – diffusion coefficient  [scales v, a, sv]
      sv           – inter-trial drift SD
      sz           – uniform starting-point range  w ~ Uniform(w-sz/2, w+sz/2)
                     integrated via 5-point Gauss-Legendre quadrature

    Reference for σ scaling:
        Blurton et al. (2017), J. Math. Psych. 76, 7-12  (eq. 1 footnote)
    Reference for sz quadrature approach:
        Ratcliff & Tuerlinckx (2002), Psychon. Bull. Rev. 9, 438-481
        (HDDM and fddm both implement sz this way in practice)
    """
    t_eff = t - t0
    if t_eff <= 0.0:
        return 1e-300

    # -------------------------------------------------------
    # σ handling: pure rescaling (σ is a scale parameter)
    # v/σ, a/σ, sv/σ are the normalised parameters
    # -------------------------------------------------------
    if sigma != 1.0:
        v  = v  / sigma
        a  = a  / sigma
        sv = sv / sigma

    # -------------------------------------------------------
    # sz = 0 → no starting-point variability, fast path
    # -------------------------------------------------------
    if sz <= 1e-10:
        return _ddm_pdf_lower_core(t_eff, v, a, w, sv, max_J, eps)

    # -------------------------------------------------------
    # sz > 0: Gauss-Legendre quadrature over
    #   w' ∈ [w - sz/2, w + sz/2]  (clipped to (0,1))
    #
    # (1/sz) ∫_{w-sz/2}^{w+sz/2} f(t|w') dw'
    #  = (1/2) Σ_i weight_i * f(t | w + (sz/2)*node_i)
    # -------------------------------------------------------
    # Hard-coded 5-point GL nodes and weights (on [-1, 1])
    n0 = -0.9061798459386640;  wt0 = 0.2369268850561891
    n1 = -0.5384693101056831;  wt1 = 0.4786286704993665
    n2 =  0.0;                 wt2 = 0.5688888888888889
    n3 =  0.5384693101056831;  wt3 = 0.4786286704993665
    n4 =  0.9061798459386640;  wt4 = 0.2369268850561891

    half_sz = sz * 0.5
    acc = 0.0

    for (node, wt) in ((n0, wt0), (n1, wt1), (n2, wt2), (n3, wt3), (n4, wt4)):
        w_prime = w + half_sz * node

        # Clip to valid range – avoid boundary singularities
        if w_prime <= 0.0:
            w_prime = 1e-6
        elif w_prime >= 1.0:
            w_prime = 1.0 - 1e-6

        acc += wt * _ddm_pdf_lower_core(t_eff, v, a, w_prime, sv, max_J, eps)

    pdf = 0.5 * acc          # factor 1/2 from change-of-variables
    return max(pdf, 1e-300)

def complete_run(data, params,weights):
    vE,vH,aA,aS,t0,w,k,weight_w,weight_v,lapse,strength_bias = params


    w_part_1 = np.asarray(weights)
    w_part_2 = np.ones_like(weights) - np.flip(weights)
    weigths_use = np.concatenate((w_part_1,w_part_2))



    memorySeq = np.asarray(give_all_comb(k))

    memoryStrengthPlus = np.asarray([p*strength_bias for p in weigths_use])
    memoryStrengthMinus = np.asarray([(1-p)*strength_bias for p in weigths_use])

    p_pos_pre = [homm_get_next_w(seq, memorySeq,memoryStrengthPlus, memoryStrengthMinus, k, 0.0) for seq in memorySeq]
    

    return complete_run_fast(
        data["ease"].to_numpy(np.int8),
        data["sa"].to_numpy(np.int8),
        data["stim"].to_numpy(np.int8),
        data["resp"].to_numpy(np.int8),
        data["rt"].to_numpy(np.float64),
        p_pos_pre,
        memorySeq, memoryStrengthPlus,memoryStrengthMinus,
        vE,vH,aA,aS, t0, w, k,
        weight_w, weight_v, lapse
    )


@njit
def _sigmoid(x):
    if x >= 0:
        return 1.0 / (1.0 + np.exp(-x))
    ex = np.exp(x)
    return ex / (1.0 + ex)

@njit
def conf_2dsd(v_signed, a, resp, tau, sigma=1.0):

    mean_tc = a + resp * v_signed * tau
    sd = sigma * math.sqrt(tau)
    z = mean_tc / sd
    return 0.5 * (1.0 + math.erf(z * 0.7071067811865476))

@njit
def complete_run_fast(ease,sa, stim, resp, rt,
                      p_pos_pre,
                      memorySeq, memoryStrengthPlus,memoryStrengthMinus,
                      vE,vH,aA,aS, t0, w, k,
                      weight_w, weight_v, lapse):
    n = len(stim)
    likelihood = 0.0


    for i in range(n):
        if sa[i]==1:
            a = aS
        else:
            a = aA
        
        
        context = build_context(resp, i, k)
        
        p_pos = p_pos_pre[seq_to_index(context)]
        


        
        #print(f"this is the first {context}")

        pred_correct_dir = p_pos if stim[i] == 1 else (1.0 - p_pos)
        w_logit = np.log(w / (1.0 - w))
        w_up = _sigmoid(w_logit + weight_w * (2*p_pos - 1))

        # Same for sz
        w_sz = 0# np.sqrt(weight_v**2 * (4*p_pos*(1-p_pos)) * tau)/a


        if ease[i] == 2:
            v = vE + weight_v*(pred_correct_dir - (1-pred_correct_dir))

        else:
            v = vH + weight_v*(pred_correct_dir - (1-pred_correct_dir))

        if stim[i] == 1:
            v_eff = v  if resp[i] == 1  else -v
            w_eff = w_up if resp[i] == 1 else 1.0 - w_up
        else:
            v_eff = v  if resp[i] == -1 else -v
            w_eff = (1.0 - w_up) if resp[i] == -1 else w_up
        

        p_ddm   = ddm_pdf_lower_boundary(rt[i], -v_eff, a, 1.0-w_eff, t0,sz = w_sz)
        p_lapse = 0.5 / (4.0 - 0.15)
        p = (1.0 - lapse)*p_ddm + lapse*p_lapse
        if p > 0:
            likelihood += math.log(p)

    return likelihood

In [97]:
[1,2,3,4][-3:]

[2, 3, 4]

In [ ]:
import numpy as np
import cma

# ── Indices into the 13 non-weight params ──────────────────────────────────
DDM_IDX     = [0, 1, 2, 3, 4, 5, 8]      # vE, vH, aA, aS, t0, w, lapse, decay
MEMORY_IDX  = [6, 7, 9, 10]               # weight_w, weight_v, gamma, bias_strength, ptau

# Neutral unit-cube values for frozen memory params during stage 1
# These map to: weight_w≈0, weight_v≈0, gamma=0, bias_strength≈0, ptau=middle
MEMORY_NEUTRAL = [0.0, 0.0, 0.0, 0.0]

def make_rescale(bl, bh, ls):
    safe_bl = np.where(ls, bl, 1.0)
    safe_bh = np.where(ls, bh, 1.0)
    def rescale(x_unit):
        x = np.asarray(x_unit)
        log_vals = safe_bl * (safe_bh / safe_bl) ** x
        lin_vals = bl + x * (bh - bl)
        return np.where(ls, log_vals, lin_vals)
    return rescale


def fit_ddm_stage(df_participant, rescale, bounds_low, bounds_high, n_w,
                  n_restarts=5, maxiter=400):
    """Stage 1: fit only DDM params with memory frozen to neutral values."""
    n_params = len(bounds_low)

    def neg_log_lik_ddm(x_ddm_unit):
        # Build full unit-cube vector with memory frozen
        x_full = np.zeros(n_params)
        x_full[DDM_IDX]    = x_ddm_unit[:len(DDM_IDX)]
        x_full[MEMORY_IDX] = MEMORY_NEUTRAL
        x_full[10:]        = 0.5    # weights at neutral
        params = rescale(np.clip(x_full, 0.0, 1.0))
        vE, vH, aA, aS, t0, w, weight_w, weight_v, lapse, bias_strength = params[:10]
        weights = params[10:]
        full_params = (vE, vH, aA, aS, t0, w, 1,   # k=1 for stage 1
                       weight_w, weight_v, lapse, bias_strength)
        ll = -complete_run(df_participant, full_params, weights)
        return ll if np.isfinite(ll) else 1e10

    n_ddm = len(DDM_IDX)
    best_val    = np.inf
    best_x_ddm  = None

    for _ in range(n_restarts):
        x0     = np.random.uniform(0.0, 1.0, n_ddm)
        sigma0 = 0.3

        opts = cma.CMAOptions()
        opts['bounds']        = [[0.0]*n_ddm, [1.0]*n_ddm]
        opts['maxiter']       = maxiter
        opts['popsize']       = 15
        opts['tolstagnation'] = 30
        opts['tolfun']        = 1e-5
        opts['tolx']          = 1e-5
        opts['verbose']       = -9
        opts['seed']          = np.random.randint(0, 100000)


        es = cma.CMAEvolutionStrategy(x0, sigma0, opts)
        es.optimize(neg_log_lik_ddm)
        result = es.result
        if np.isfinite(result.fbest) and result.fbest < best_val:
            best_val   = result.fbest
            best_x_ddm = np.clip(result.xbest, 0.0, 1.0)


    print(f"  Stage 1 DDM fit: LL = {-best_val:.2f}")
    return best_x_ddm   # unit-cube values for DDM params only

def fit_mle(df_participant, n_restarts=8, k_values=range(1, 6)):

    best_overall = {"params": None, "k": None, "neg_ll": np.inf, "aic": np.inf, "param_names": None}
    print(f"Running optimization for k = {list(k_values)}")

    # ── Stage 1: run ONCE, outside k-loop, always k=1 ─────────────────────
    n_w_s1       = 1
    bounds_low_s1  = np.array([0.1, 0.1, 0.1, 0.1, 0.0, 0.3, 0.000001, 0.0000001, 0.0, 0.00001] + [0.0]*n_w_s1)
    bounds_high_s1 = np.array([9.0, 9.0, 3.0, 3.0, 0.5, 0.7, 100.0,   100.0,    1.0, 100000.0] + [1.0]*n_w_s1)
    log_scale_s1   = np.array([False, False, False, False, False, False, True, False, False,True] + [False]*n_w_s1)
    rescale_s1     = make_rescale(bounds_low_s1, bounds_high_s1, log_scale_s1)

    print("Running Stage 1 (DDM only)...")
    ddm_x_unit = fit_ddm_stage(df_participant, rescale_s1, bounds_low_s1, bounds_high_s1, n_w_s1)

    # ── Stage 2: k-loop with full fits ────────────────────────────────────
    for k in k_values:
        k_w = min(5, k) - 1
        n_w = 2**k_w

        bounds_low_k  = np.array([0.1, 0.1, 0.1, 0.1, 0.0, 0.3, 0.000001, 0.0000001, 0.0, 0.00001] + [0.0]*n_w)
        bounds_high_k = np.array([9.0, 9.0, 3.0, 3.0, 0.5, 0.7, 100.0,   100.0,    1.0, 100000.0] + [1.0]*n_w)
        log_scale_k   = np.array([False, False, False, False, False, False, True, False, False, True] + [False]*n_w)

        param_names = ["vE","vH","aA","aS","t0","w","weight_w","weight_v","lapse",
                       "bias_strength"] + ["w" + str(i) for i in range(n_w)]
        n_params = len(bounds_low_k)

        print(f"\n=== Optimizing with fixed k = {k} (n_params={n_params}) ===")

        rescale = make_rescale(bounds_low_k, bounds_high_k, log_scale_k)

        def neg_log_lik(params_unit):
            params = rescale(np.clip(params_unit, 0.0, 1.0))
            vE, vH, aA, aS, t0, w, weight_w, weight_v, lapse, bias_strength = params[:10]
            weights = params[10:]
            full_params = (vE, vH, aA, aS, t0, w, k, weight_w, weight_v, lapse, bias_strength)
            ll = -complete_run(df_participant, full_params, weights)
            return ll if np.isfinite(ll) else 1e10

        best_k_params = None
        best_k_val    = np.inf
        successful_fits = 0

        print("  Running Stage 2 (full fit)...")
        for i in range(n_restarts):
            x0 = np.random.uniform(0.0, 1.0, n_params)
            if i % 4 != 0 and ddm_x_unit is not None:
                x0[DDM_IDX]    = np.clip(ddm_x_unit + np.random.normal(0, 0.05, len(DDM_IDX)), 0.0, 1.0)
                x0[MEMORY_IDX] = np.random.uniform(0.0, 1.0, len(MEMORY_IDX))

            sigma0 = 0.3
            opts = cma.CMAOptions()
            opts['bounds']        = [[0.0]*n_params, [1.0]*n_params]
            opts['maxiter']       = 600 + 200*k
            opts['popsize']       = 10
            opts['tolstagnation'] = 50 + 10*k_w
            opts['tolfun']        = 1e-3
            opts['tolx']          = 1e-3
            opts['verbose']       = -9
            opts['seed']          = np.random.randint(0, 100000)
            opts['CMA_stds']      = [1.0] * n_params

            try:
                es = cma.CMAEvolutionStrategy(x0, sigma0, opts)
                es.optimize(neg_log_lik)
                result = es.result
                if np.isfinite(result.fbest):
                    successful_fits += 1
                    if result.fbest < best_k_val:
                        best_k_val    = result.fbest
                        best_k_params = rescale(np.clip(result.xbest, 0.0, 1.0))
                        print(f"  Restart {i+1} ({'warm' if i%2==1 else 'random'}): "
                              f"LL = {-best_k_val:.2f}  [{result.stop}]")
            except Exception as e:
                print(f"  Restart {i+1} failed: {e}")
                continue

        print(f"  Successful fits for k={k}: {successful_fits}/{n_restarts + k}")

        if best_k_params is not None:
            k_aic = 2 * n_params + 2 * best_k_val
            print(f"  AIC for k={k}: {k_aic:.2f}  (vs current best: {best_overall['aic']:.2f})")
            if k_aic < best_overall["aic"]:
                best_overall["neg_ll"]      = best_k_val
                best_overall["aic"]         = k_aic
                best_overall["params"]      = np.asarray(list(best_k_params) + [-1.0] * (2**4 - n_w))
                best_overall["k"]           = k
                best_overall["param_names"] = param_names

    if best_overall["params"] is not None:
        print("\n=== Best overall fit (by AIC) ===")
        print(f"k = {best_overall['k']}")
        for name, val in zip(best_overall["param_names"], best_overall["params"]):
            print(f"  {name}: {val:.4f}")
        print(f"Final Log-Likelihood : {-best_overall['neg_ll']:.2f}")
        print(f"Final AIC            : {best_overall['aic']:.2f}")
        return best_overall["params"], best_overall["k"], -best_overall["neg_ll"], best_overall["aic"]
    else:
        print("No successful fits found!")
        return None, None, None, None

In [99]:
[1,2,3][:-2]

[1]

In [100]:



def fit_all_participants(df_path, output_path=None, test_first_n=None):
    """Fit RDM to all participants"""
    
    # Load data
    df = pd.read_csv(df_path)
    df["resp"] = df["resp"].replace({"R":1, "L":-1})
    df["stim"] = df["stim"].replace({"R":1, "L":-1})


    print(f"Loaded data with {len(df)} rows")
    print(f"Unique participants: {df['pp'].nunique()}")
    print(f"Columns: {list(df.columns)}")
    
    # Check data structure
    print("\nData preview:")
    print(df.head())
    
    # Initialize results list
    results = []
    
    # Get participant list
    participants = df['pp'].unique()
    if test_first_n:
        participants = participants[:test_first_n]
        print(f"Testing first {test_first_n} participants only")

    
    # Loop over participants
    for i, participant_id in enumerate(participants):
        print(f"\n{'='*50}")
        print(f"Fitting participant {participant_id} ({i+1}/{len(participants)})")
        print(f"{'='*50}")
        
        participant_df = df[df['pp'] == participant_id]
        print(f"Participant has {len(participant_df)} trials")
        

        params,k, max_ll,aic = fit_mle(participant_df, n_restarts=7)  # Fewer restarts for testing
        
        if params is not None:
            results.append([participant_id] + list(params) + [k] + [max_ll] + [aic])
            print(f"✓ Success: LL = {max_ll:.2f}")
        else:
            results.append([participant_id] + [None] * 11 + [None])
            print("✗ Failed: No valid fit found")
                
    # Create results DataFrame
    columns = ["id","vE","vH","aA","aS","t0","w","weight_w","weight_v","lapse","bias_strength"]+["w" + str(i) for i in range(2**4)]+["k", "max_log_lik","aic"]

    results_df = pd.DataFrame(results, columns=columns)

    print(f"\nParameter summary (successful fits only):")
    numeric_cols = columns

    print(results_df[numeric_cols].describe())
    
    # Save results
    if output_path:
        results_df.to_csv(output_path, index=False)
        print(f"\nResults saved to: {output_path}")
    
    return results_df

# Example usage:
if __name__ == "__main__":
    # Test with first few participants only
    results_df = fit_all_participants(
        df_path="../raw_factorial_data.csv",
        output_path="SPRB.csv",
        test_first_n=25  # Test with first 3 participants only
    )
    
    print("\nFirst few results:")
    print(results_df)

/var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/ipykernel_56232/1344660385.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["resp"] = df["resp"].replace({"R":1, "L":-1})
/var/folders/g5/d_krhx3n0xg4nf8x8frstwdw0000gq/T/ipykernel_56232/1344660385.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["stim"] = df["stim"].replace({"R":1, "L":-1})


Loaded data with 56097 rows
Unique participants: 20
Columns: ['pp', 'trial', 'block', 'block.trial', 'ease', 'sa', 'bias', 'fix.dur', 'stim', 'resp', 'correct', 'rt']

Data preview:
   pp  trial  block  block.trial  ease  sa bias  fix.dur  stim  resp  correct  \
0   1      1      6            1     2   1   no    0.738     1     1        1   
1   1      2      6            2     1   1   no    0.784     1     1        1   
2   1      3      6            3     1   1   no    0.651     1     1        1   
3   1      4      6            4     1   1   no    0.781     1     1        1   
4   1      5      6            5     1   1   no    0.585    -1     1        0   

      rt  
0  0.482  
1  0.602  
2  0.381  
3  0.584  
4  0.464  
Testing first 25 participants only

Fitting participant 1 (1/20)
Participant has 2807 trials
Running optimization for k = [1, 2, 3, 4, 5]
Running Stage 1 (DDM only)...
  Stage 1 DDM fit: LL = -1134.86

=== Optimizing with fixed k = 1 (n_params=11) ===
  Running Sta

In [101]:
df = pd.read_csv("SPRBLaicMoreSym.csv")
df2 = pd.read_csv("SPRBLnoGamma.csv")
df3 = pd.read_csv("/Users/Anton/Desktop/BachelorProjekt/OtherData/GillesDutilh/MoreSymAndFrund/FRUND.csv")
df4 = pd.read_csv("/Users/Anton/Desktop/BachelorProjekt/OtherData/GillesDutilh/ForFirstDraft/PureDDM.csv")
aic_df3 = 2*(8 +4*df3["k"]) - 2 * df3["max_log_lik"]
aic_df4 = 2*(len(df4.columns)-3) - 2 * df4["max_log_lik"]


aic_df4 - df["aic"]


0       3.631994
1     243.073387
2     364.399721
3     116.551043
4     923.796101
5     171.658885
6     209.069441
7      74.831397
8     134.498339
9      82.413125
10     16.773387
11     -0.263575
12     69.774783
13     76.648051
14     38.814163
15     27.633232
16      3.891884
17     -2.268912
18    218.236856
19     99.188992
dtype: float64

In [102]:
df2["weight_v"]

0     0.286667
1     3.345838
2     3.022308
3     1.759145
4     1.495829
5     0.724318
6     0.000001
7     2.822184
8     1.774475
9     2.110596
10    3.132864
11    0.060183
12    0.000010
13    0.000030
14    0.530088
15    0.125121
16    0.000006
17    0.353261
18    6.776630
19    0.000002
Name: weight_v, dtype: float64

In [103]:
df2["k"]

0     3
1     3
2     4
3     4
4     5
5     4
6     4
7     4
8     4
9     4
10    2
11    3
12    3
13    3
14    2
15    5
16    2
17    3
18    4
19    5
Name: k, dtype: int64

In [104]:

weights = np.asarray(weights)
weights = 2*weights/np.sum(weights)
print(weights)
w_correction = np.asarray([max(0,i) for i in weights-np.ones_like(weights)])

to_correct = w_correction<=0
weights = weights - w_correction
correction = 0 if len(w_correction[to_correct]) == 0 else sum(w_correction)/len(w_correction[to_correct])

weights = np.asarray([correction + i if to_correct else i for to_correct,i in zip(to_correct,weights)])

weights,sum(weights)

NameError: name 'weights' is not defined

In [ ]:
for i in (range(2,4)):
    print(i)

2
3


In [ ]:
print(give_all_comb(3))
round(df[df.columns[13:-3]],2)

[(-1, -1, -1), (-1, -1, 1), (-1, 1, -1), (-1, 1, 1), (1, -1, -1), (1, -1, 1), (1, 1, -1), (1, 1, 1)]


,w1,w2,w3,w4,w5,w6,w7
0,0.23,0.20,0.73,0.66,0.44,0.24,0.51
1,0.00,0.16,0.88,1.00,0.00,0.96,0.00
2,0.04,0.00,1.00,0.95,0.01,0.11,1.00
3,0.11,0.66,0.88,0.45,0.00,0.42,0.63
4,0.28,0.94,0.00,1.00,0.00,0.78,0.00
5,0.04,0.62,0.15,1.00,0.04,0.23,0.97
6,0.46,0.00,0.74,0.55,0.52,0.44,0.67
7,0.00,0.00,0.97,0.48,0.71,0.25,0.59
8,0.00,0.34,0.76,1.00,0.00,0.26,0.76
9,0.21,0.40,0.65,0.33,0.26,0.28,0.88


In [ ]:
import pandas as pd 
df =  pd.read_csv("PriorDecayAndNoBS.csv")
df["ws_total"] = df.apply(lambda row: sum(row[df.columns[13:-3]]),axis = 1)
# for w in df.columns[13:-3]:
#     df[w] = df[w]*4/df["ws_total"]

df["ws_total"] = df.apply(lambda row: sum(row[df.columns[13:-3]]),axis = 1)
round(df[["weight_w"]+["w"]+["bias_strength"]+["ws_total"]+list(df.columns[13:-3])],2)

FileNotFoundError: [Errno 2] No such file or directory: 'PriorDecayAndNoBS.csv'

In [ ]:
round(df[df.columns[12:]],2)

NameError: name 'df' is not defined